# Riemann data pipeline — consolidated 00–05

Single Colab notebook combining environment, acquisition, description, unfolding, surrogate generation, and comparison. Existing notebooks remain unchanged.

The pipeline is ordered so each stage produces the objects/files consumed by the next stage.

## 00 — Environment

In [ ]:
from pathlib import Path
import sys
import numpy as np

REPO_BASE = Path("/content/nicht-riemann-data")
if REPO_BASE.exists():
    %cd /content/nicht-riemann-data

print(sys.version)
print("numpy:", np.__version__)
print("cwd:", Path.cwd())

In [ ]:
# Install the project when running from a fresh Colab clone.
# If already installed, pip simply verifies the editable installation.
!{sys.executable} -m pip install -e .

In [ ]:
from nicht_riemann_data.transforms import spacings
from nicht_riemann_data.diagnostics import describe
print("Package import: OK")

## 01 — Acquire

In [ ]:
import hashlib
import urllib.request

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ODLYZKO_BASE = "https://www-users.cse.umn.edu/~odlyzko/zeta_tables"
DATASET = "zeros1"
URL = f"{ODLYZKO_BASE}/{DATASET}"
RAW_FILE = DATA_DIR / DATASET
print("URL :", URL)
print("file:", RAW_FILE)

In [ ]:
if RAW_FILE.exists():
    print(f"Already exists: {RAW_FILE}")
    print(f"Bytes: {RAW_FILE.stat().st_size:,}")
else:
    urllib.request.urlretrieve(URL, RAW_FILE)
    print(f"Downloaded: {RAW_FILE}")
    print(f"Bytes: {RAW_FILE.stat().st_size:,}")

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("SHA-256:", sha256(RAW_FILE))

In [ ]:
gamma = np.loadtxt(RAW_FILE, dtype=np.float64)
assert gamma.ndim == 1
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)
print("N =", len(gamma))
print("first =", gamma[:5])
print("last  =", gamma[-5:])

In [ ]:
delta = spacings(gamma)
assert len(delta) == len(gamma) - 1
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)
print("zeros    :", len(gamma))
print("spacings :", len(delta))
print("first    :", delta[:10])
print("gamma :")
describe(gamma)
print("delta :")
describe(delta)

## 02 — Describe

In [ ]:
gamma_range = gamma[-1] - gamma[0]
mean_spacing = np.mean(delta)
range_per_spacing = gamma_range / len(delta)
print("range:", gamma_range)
print("mean spacing:", mean_spacing)
print("range / number of spacings:", range_per_spacing)
assert np.isclose(mean_spacing, range_per_spacing)

In [ ]:
percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
percentiles = np.percentile(delta, percentile_levels)
for p, value in zip(percentile_levels, percentiles):
    print(f"{p:>3}% : {value:.12f}")

In [ ]:
BLOCK_SIZE = 1000
num_blocks = len(delta) // BLOCK_SIZE
block_means = np.array([np.mean(delta[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE]) for i in range(num_blocks)])
remainder_delta = delta[num_blocks * BLOCK_SIZE:]
print("block size:", BLOCK_SIZE)
print("full blocks:", num_blocks)
print("remainder:", len(remainder_delta))
print("first block means:")
print(block_means[:10])
print("last block means:")
print(block_means[-10:])
print("local mean spacing:")
print("  min :", block_means.min())
print("  max :", block_means.max())
print("  mean:", block_means.mean())
print("  std :", block_means.std())
print("ratio max/min:", block_means.max() / block_means.min())

In [ ]:
local_residuals = np.empty_like(delta)
for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE
    local_residuals[start:end] = delta[start:end] - block_means[i]
if len(remainder_delta):
    start = num_blocks * BLOCK_SIZE
    local_residuals[start:] = remainder_delta - np.mean(remainder_delta)
assert local_residuals.shape == delta.shape
print("local-mean residual:")
print("  std :", np.std(local_residuals))
print("  min :", np.min(local_residuals))
print("  max :", np.max(local_residuals))

In [ ]:
predicted_global = gamma[0] + np.arange(len(gamma)) * mean_spacing
residual_global = gamma - predicted_global
print("global constant-spacing baseline:")
print("  residual std:", np.std(residual_global))
print("  residual min:", np.min(residual_global))
print("  residual max:", np.max(residual_global))

In [ ]:
predicted_delta = np.empty_like(delta)
for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE
    predicted_delta[start:end] = block_means[i]
if len(remainder_delta):
    predicted_delta[num_blocks * BLOCK_SIZE:] = np.mean(remainder_delta)
residual = delta - predicted_delta
assert predicted_delta.shape == delta.shape
assert residual.shape == delta.shape
print("local block baseline:")
print("  residual std:", np.std(residual))
print("  residual min:", np.min(residual))
print("  residual max:", np.max(residual))

## 03 — Unfold

Local unfolding uses the same 1000-spacing block baseline as the exploratory description above. The derived array is persisted so later stages do not depend on notebook state.

In [ ]:
local_mean = np.empty_like(delta)
for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE
    local_mean[start:end] = np.mean(delta[start:end])
remainder_start = num_blocks * BLOCK_SIZE
if remainder_start < len(delta):
    local_mean[remainder_start:] = np.mean(delta[remainder_start:])
unfolded = delta / local_mean
assert unfolded.shape == delta.shape
assert np.all(np.isfinite(unfolded))
assert np.all(unfolded > 0)
print("unfolded:", len(unfolded))
print("mean:", np.mean(unfolded))
print("std :", np.std(unfolded))

In [ ]:
DERIVED_DIR = Path("data/derived")
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
UNFOLDED_FILE = DERIVED_DIR / "unfolded_spacings.float64"
unfolded.astype(np.float64).tofile(UNFOLDED_FILE)
print("saved:", UNFOLDED_FILE)
print("bytes:", UNFOLDED_FILE.stat().st_size)

## 04 — Surrogates

In [ ]:
SEED = 20260831
rng = np.random.default_rng(SEED)
print("seed:", SEED)

surrogate_shuffle = unfolded.copy()
rng.shuffle(surrogate_shuffle)
assert np.array_equal(np.sort(surrogate_shuffle), np.sort(unfolded))
print("shuffle surrogate created")
print("same values:", np.array_equal(np.sort(surrogate_shuffle), np.sort(unfolded)))

In [ ]:
surrogate_iid = rng.choice(unfolded, size=len(unfolded), replace=True)
assert surrogate_iid.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_iid))
assert np.all(surrogate_iid > 0)
print("iid surrogate created")
print("N:", len(surrogate_iid))

surrogate_uniform = rng.uniform(0.0, 2.0, size=len(unfolded))
assert surrogate_uniform.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_uniform))
assert np.all(surrogate_uniform >= 0)
print("uniform surrogate created")
print("N:", len(surrogate_uniform))
print("mean:", np.mean(surrogate_uniform))

In [ ]:
datasets = {
    "observed": unfolded,
    "shuffle": surrogate_shuffle,
    "iid": surrogate_iid,
    "uniform": surrogate_uniform,
}
for name, values in datasets.items():
    print(f"{name:>8} : mean={np.mean(values):.6f}  std={np.std(values):.6f}  min={np.min(values):.6f}  max={np.max(values):.6f}")

In [ ]:
percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
for name, values in datasets.items():
    print(f"\n{name}")
    for p, value in zip(percentile_levels, np.percentile(values, percentile_levels)):
        print(f"{p:>3}% : {value:.12f}")

## 05 — Compare

Comparison only: no new acquisition, unfolding, or surrogate generation. In the consolidated notebook the required objects are already present from 03–04.

In [ ]:
required = {"shuffle": surrogate_shuffle, "iid": surrogate_iid, "uniform": surrogate_uniform}
datasets = {"observed": unfolded, **required}
for name, values in datasets.items():
    assert values.ndim == 1
    assert len(values) == len(unfolded)
    assert np.all(np.isfinite(values))
    assert np.all(values >= 0)
print("datasets:", ", ".join(datasets))
print("N:", len(unfolded))

In [ ]:
print("=== distribution summary ===")
for name, values in datasets.items():
    print(f"{name:>8}: mean={np.mean(values):.8f}  std={np.std(values):.8f}  min={np.min(values):.8f}  max={np.max(values):.8f}")

print("=== quantiles ===")
PERCENTILES = [1, 5, 25, 50, 75, 95, 99]
for name, values in datasets.items():
    q = np.percentile(values, PERCENTILES)
    print(name, {p: float(v) for p, v in zip(PERCENTILES, q)})

In [ ]:
def lag1_correlation(values):
    return np.corrcoef(values[:-1], values[1:])[0, 1]

print("=== lag-1 correlation ===")
for name, values in datasets.items():
    print(f"{name:>8}: {lag1_correlation(values):+.8f}")

def block_means(values, block_size=BLOCK_SIZE):
    n = len(values) // block_size
    return np.asarray([np.mean(values[i * block_size:(i + 1) * block_size]) for i in range(n)])

print("=== block-mean variation ===")
for name, values in datasets.items():
    means = block_means(values)
    print(f"{name:>8}: blocks={len(means)}  min={means.min():.8f}  max={means.max():.8f}  std={means.std():.8f}")

In [ ]:
assert np.array_equal(np.sort(surrogate_shuffle), np.sort(unfolded))
assert np.isclose(np.mean(surrogate_shuffle), np.mean(unfolded))
assert np.isclose(np.std(surrogate_shuffle), np.std(unfolded))
print("\nALL COMPARISON INPUTS AND BASIC INVARIANTS PASSED")